<a href="https://colab.research.google.com/github/Tseng0318/peft/blob/main/Model_Merging_GT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Merging — Inference with Ground Truth
Uses `allenai/ai2_arc` (ARC-Challenge) for science and `cais/mmlu` (high_school_mathematics) for math.
Both datasets are MCQA with ground truth answer keys, so accuracy can be computed directly.

In [18]:
!nvidia-smi

Sat Apr 18 19:28:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   36C    P0             89W /  600W |    5447MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install Packages

In [ ]:
!pip install -q --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
!pip install -q datasets==3.5.1 bitsandbytes>=0.46.1 huggingface_hub>=1.3.0

## Import Packages

In [19]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

import json, os, re
import numpy as np
import torch
from tqdm import tqdm
from datasets import load_dataset
from peft import PeftModel, LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


## Load Datasets with Ground Truth
- **Math**: `cais/mmlu` → `high_school_mathematics` test split (100 questions, MCQA, answerKey 0-3)
- **Science**: `allenai/ai2_arc` → `ARC-Challenge` test split (MCQA, answerKey A-D)

In [20]:
N_SAMPLES = 200  # set to None to use the full test split

# ── Math: MMLU high school mathematics ──
mmlu_raw = load_dataset("cais/mmlu", "high_school_mathematics", split="test")
if N_SAMPLES:
    mmlu_raw = mmlu_raw.select(range(min(N_SAMPLES, len(mmlu_raw))))

OPTION_LABELS = ["A", "B", "C", "D"]
math_data = []
for i, item in enumerate(mmlu_raw):
    math_data.append({
        "id": f"mmlu_math_{i}",
        "question": item["question"],
        "options": {OPTION_LABELS[j]: item["choices"][j] for j in range(4)},
        "answer": OPTION_LABELS[item["answer"]],  # ground truth
    })

# ── Science: ARC-Challenge ──
arc_raw = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
if N_SAMPLES:
    arc_raw = arc_raw.select(range(min(N_SAMPLES, len(arc_raw))))

science_data = []
for i, item in enumerate(arc_raw):
    labels = item["choices"]["label"]
    texts  = item["choices"]["text"]
    # Normalize labels: some ARC items use 1/2/3/4 instead of A/B/C/D
    norm = {"1":"A","2":"B","3":"C","4":"D"}
    options = {norm.get(l, l): t for l, t in zip(labels, texts)}
    ak = item["answerKey"]
    ak = norm.get(ak, ak)
    science_data.append({
        "id": f"arc_{i}",
        "question": item["question"],
        "options": options,
        "answer": ak,  # ground truth
    })

print(f"Math (MMLU)   : {len(math_data)} questions")
print(f"Science (ARC) : {len(science_data)} questions")
print(f"\nSample Math   : {math_data[0]}")
print(f"\nSample Science: {science_data[0]}")

Math (MMLU)   : 200 questions
Science (ARC) : 200 questions

Sample Math   : {'id': 'mmlu_math_0', 'question': 'If a pentagon P with vertices at (– 2, – 4), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across the line y = x to get a new pentagon, P’, then one of the vertices of P’ is', 'options': {'A': '(0, – 3)', 'B': '(4, 1)', 'C': '(2, 2)', 'D': '(– 4, –2)'}, 'answer': 'D'}

Sample Science: {'id': 'arc_0', 'question': 'An astronomer observes that a planet rotates faster after a meteorite impact. Which is the most likely effect of this increase in rotation?', 'options': {'A': 'Planetary density will decrease.', 'B': 'Planetary years will become longer.', 'C': 'Planetary days will become shorter.', 'D': 'Planetary gravity will become stronger.'}, 'answer': 'C'}


## Save Datasets Locally

In [21]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/merge_result"
os.makedirs(DRIVE_DIR, exist_ok=True)

with open(f"{DRIVE_DIR}/MATH.json", "w") as f:
    json.dump(math_data, f, indent=2)
with open(f"{DRIVE_DIR}/SCIENCE.json", "w") as f:
    json.dump(science_data, f, indent=2)
print(f"Saved MATH.json and SCIENCE.json → {DRIVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved MATH.json and SCIENCE.json → /content/drive/MyDrive/merge_result


## Prompt Generation

In [22]:
INSTRUCTION = {
    "MATH":    "You are given a math question and four answer options (associated with \"A\", \"B\", \"C\", \"D\"). Your task is to carefully analyze the problem, apply logical reasoning, and select the correct answer. There is only one correct answer for each question.",
    "SCIENCE": "You are given a science question and four answer options (associated with \"A\", \"B\", \"C\", \"D\"). Your task is to find the correct answer based on scientific facts, knowledge, and reasoning. There is only one correct answer for each question.",
}

def build_prompt(task_name, data_point):
    sys_msg = INSTRUCTION[task_name]
    opts = data_point["options"]
    options_str = " ".join([f"({k}) {v}" for k, v in opts.items()])
    user_prompt = f"Question: {data_point['question']}; Options: {options_str}"
    return f"""[INST] <<SYS>>You are a helpful assistant and good at solving tasks based on task instructions and inputs.<</SYS>> Instruction: {sys_msg} Input: {user_prompt}[/INST]"""

## Seed & Device Settings

In [23]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda:0")
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
np.random.seed(42)

## Merging Settings

In [15]:
#### adjust ####
MERGE_TYPE   = "linear"   # linear | magnitude_prune | ties | dare_ties | dare_linear
density      = 0.2
weight_math  = 1.0
weight_sci   = 0.4
weights      = [weight_math, weight_sci]


In [24]:
OUTPUT_DIR = "/content/drive/MyDrive/merge_result/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load Model & Adapters

In [25]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
lora_config = LoraConfig(
    r=8, target_modules=["q_proj","k_proj","v_proj"],
    task_type="CAUSAL_LM", lora_alpha=16, lora_dropout=0.05
)

base_model_name = "unsloth/llama-2-7b-chat-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
tokenizer.pad_token_id = 1
tokenizer.eos_token_id = 2

model = AutoModelForCausalLM.from_pretrained(
    base_model_name, quantization_config=quant_config,
    low_cpu_mem_usage=True, torch_dtype=torch.float16
).to(device)
model.resize_token_embeddings(len(tokenizer))

hf_adapters = {
    "MATH":    "MonicaHuang/llama-2-7b-chat-GSM8K-MCQA",
    "SCIENCE": "chenjoachim/llama-2-7b-chat-ARC-MCQA",
}
TASK_NAMES = ["MATH", "SCIENCE"]

model = PeftModel.from_pretrained(model, hf_adapters[TASK_NAMES[0]], adapter_name=TASK_NAMES[0]).to(device)
for t in TASK_NAMES[1:]:
    model.load_adapter(hf_adapters[t], adapter_name=t, device=device)

print("Model and adapters loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model and adapters loaded.


## Apply Merging Algorithm

In [ ]:
if MERGE_TYPE == "ties":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="ties", density=density)
elif MERGE_TYPE == "linear":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="linear")
elif MERGE_TYPE == "magnitude_prune":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="magnitude_prune", density=density)
elif MERGE_TYPE == "dare_ties":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="dare_ties", density=density)
elif MERGE_TYPE == "dare_linear":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="dare_linear", density=density)

model.set_adapter("merge")
print(f"Merged with: {MERGE_TYPE} | density={density} | weights={weights}")

Merged with: linear | density=0.2 | weights=[1.0, 0.4]


## Generation Config

In [26]:
hyperparameters = {
    "do_sample": False, "temperature": None,
    "num_beams": 1, "top_p": None,
    "no_repeat_ngram_size": 3, "max_new_len": 400
}
generation_config = GenerationConfig(
    do_sample=hyperparameters["do_sample"],
    temperature=hyperparameters["temperature"],
    top_p=hyperparameters["top_p"],
    num_beams=hyperparameters["num_beams"],
    pad_token_id=1,
    max_new_tokens=hyperparameters["max_new_len"]
)
print("Generation config set.")

Generation config set.


## Inference Helper

In [27]:
def run_inference(task_name, test_datas):
    results = []
    for data_point in tqdm(test_datas, desc=task_name):
        prompt = build_prompt(task_name, data_point)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
        input_ids = inputs["input_ids"].cuda()

        model.eval()
        with torch.no_grad():
            output = model.generate(
                input_ids=input_ids,
                generation_config=generation_config,
                return_dict_in_generate=True,
                output_scores=True,
                max_new_tokens=hyperparameters["max_new_len"],
            )
        for s in output.sequences:
            predict = tokenizer.decode(s)
        response = predict.split("[/INST]")[-1].split("</s>")[0].strip()

        results.append({
            "id":       data_point["id"],
            "response": response,
            "answer":   data_point["answer"],   # ground truth stored alongside
        })
    return results

## Run Math Inference (MMLU)

In [ ]:
wnames = '_'.join([str(float(w)) for w in weights])

In [ ]:
math_results = run_inference("MATH", math_data)

result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}/MATH"
os.makedirs(result_dir, exist_ok=True)
out_path = f"{result_dir}/w_{wnames}_d_{density}_result.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(math_results, f, indent=2, ensure_ascii=False)
print(f"Saved → {out_path}")

MATH: 100%|██████████| 200/200 [26:44<00:00,  8.02s/it]

Saved → /content/drive/MyDrive/merge_result/outputs/linear/MATH/w_1.0_0.4_d_0.2_result.json


## Run Science Inference (ARC-Challenge)

In [ ]:
science_results = run_inference("SCIENCE", science_data)



SCIENCE: 100%|██████████| 200/200 [19:46<00:00,  5.93s/it]


In [ ]:
result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}/SCIENCE"
os.makedirs(result_dir, exist_ok=True)
out_path = f"{result_dir}/w_{wnames}_d_{density}_result.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(science_results, f, indent=2, ensure_ascii=False)
print(f"Saved → {out_path}")

Saved → /content/drive/MyDrive/merge_result/outputs/linear/SCIENCE/w_1.0_0.4_d_0.2_result.json


## MATH adapter baseline:

In [ ]:
model.set_adapter("MATH")

math_only = run_inference("MATH", math_data)

result_dir = f"{OUTPUT_DIR}/MATH_only/MATH"
os.makedirs(result_dir, exist_ok=True)
with open(f"{result_dir}/w_1.0_d_1.0_result.json", "w") as f:
    json.dump(math_only, f, indent=2)

MATH:   0%|          | 0/200 [00:06<?, ?it/s]


KeyboardInterrupt: 

## SCIENCE adapter baseline

In [ ]:
model.set_adapter("SCIENCE")

science_only = run_inference("SCIENCE", science_data)

result_dir = f"{OUTPUT_DIR}/SCIENCE_only/SCIENCE"
os.makedirs(result_dir, exist_ok=True)
with open(f"{result_dir}/w_1.0_d_1.0_result.json", "w") as f:
    json.dump(science_only, f, indent=2)

SCIENCE: 100%|██████████| 200/200 [03:44<00:00,  1.12s/it]


## Compute Accuracy

In [28]:
def extract_answer(response):
    matches = re.findall(r'\(([A-D])\)', response)
    return matches[-1] if matches else None

def accuracy(results):
    correct = no_pred = 0
    for r in results:
        pred = extract_answer(r["response"])
        if pred is None:
            no_pred += 1
        correct += (pred == r["answer"])
    total = len(results)
    return correct / total, correct, total, no_pred

def load(path):
    with open(path) as f:
        return json.load(f)

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/merge_result/outputs"
MERGE_TYPE = "linear"
density    = 0.2
weights    = [1.0, 0.4]
wnames     = '_'.join([str(float(w)) for w in weights])

math_results  = load(f"{OUTPUT_DIR}/{MERGE_TYPE}/MATH/w_{wnames}_d_{density}_result.json")
sci_results   = load(f"{OUTPUT_DIR}/{MERGE_TYPE}/SCIENCE/w_{wnames}_d_{density}_result.json")
math_only     = load(f"{OUTPUT_DIR}/MATH_only/MATH/w_1.0_d_1.0_result.json")
science_only  = load(f"{OUTPUT_DIR}/SCIENCE_only/SCIENCE/w_1.0_d_1.0_result.json")

math_acc,      mc, mt, mn = accuracy(math_results)
sci_acc,       sc, st, sn = accuracy(sci_results)
math_only_acc,  _,  _,  _ = accuracy(math_only)
sci_only_acc,   _,  _,  _ = accuracy(science_only)
avg = (math_acc + sci_acc) / 2

In [ ]:
print(f"{'':25s} {'MATH':>8} {'SCIENCE':>10}")
print("-" * 45)
print(f"{'MATH adapter only':25s} {math_only_acc:>8.2%} {'N/A':>10}")
print(f"{'SCIENCE adapter only':25s} {'N/A':>8} {sci_only_acc:>10.2%}")
print(f"{'Merged (' + MERGE_TYPE + ')':25s} {math_acc:>8.2%} {sci_acc:>10.2%}")
print(f"{'Average (merged)':25s} {avg:>8.2%}")

                              MATH    SCIENCE
---------------------------------------------
MATH adapter only           22.00%        N/A
SCIENCE adapter only           N/A     62.00%
Merged (linear)             15.00%     23.50%
Average (merged)            19.25%


---
## Improvements

### 1. Tune Weights — give more weight to the stronger science adapter

In [29]:
#### TODO: giving more weight to science since it's stronger ####
MERGE_TYPE   = "magnitude_prune"  # best method so far
density      = 0.2
weight_math  = 0.6
weight_sci   = 1.0
weights      = [weight_math, weight_sci]

model.add_weighted_adapter(TASK_NAMES, weights, "merge_tuned",
                           combination_type=MERGE_TYPE, density=density)
model.set_adapter("merge_tuned")
print(f"Merged with: {MERGE_TYPE} | weights={weights} | density={density}")

Merged with: magnitude_prune | weights=[0.6, 1.0] | density=0.2


In [30]:
wnames = '_'.join([str(float(w)) for w in weights])

tuned_math    = run_inference("MATH", math_data)
tuned_science = run_inference("SCIENCE", science_data)

SCIENCE: 100%|██████████| 200/200 [12:37<00:00,  3.79s/it]


In [31]:
for task, results in [("MATH", tuned_math), ("SCIENCE", tuned_science)]:
    result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}_tuned/{task}"
    os.makedirs(result_dir, exist_ok=True)
    with open(f"{result_dir}/w_{wnames}_d_{density}_result.json", "w") as f:
        json.dump(results, f, indent=2)

tuned_math_acc,  _, _, _ = accuracy(tuned_math)
tuned_sci_acc,   _, _, _ = accuracy(tuned_science)
print(f"Tuned weights {weights} | Math: {tuned_math_acc:.2%} | Science: {tuned_sci_acc:.2%} | Avg: {(tuned_math_acc+tuned_sci_acc)/2:.2%}")

Tuned weights [0.6, 1.0] | Math: 29.50% | Science: 52.50% | Avg: 41.00%


### 2. Tune Density — keep more parameters (higher density)

In [32]:
#### TODO: try higher density ####
MERGE_TYPE   = "magnitude_prune"
density      = 0.5
weight_math  = 1.0
weight_sci   = 0.4
weights      = [weight_math, weight_sci]
wnames       = '_'.join([str(float(w)) for w in weights])

model.add_weighted_adapter(TASK_NAMES, weights, "merge_dense",
                           combination_type=MERGE_TYPE, density=density)
model.set_adapter("merge_dense")

dense_math    = run_inference("MATH", math_data)
dense_science = run_inference("SCIENCE", science_data)

for task, results in [("MATH", dense_math), ("SCIENCE", dense_science)]:
    result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}_d{density}/{task}"
    os.makedirs(result_dir, exist_ok=True)
    with open(f"{result_dir}/w_{wnames}_d_{density}_result.json", "w") as f:
        json.dump(results, f, indent=2)

dense_math_acc, _, _, _ = accuracy(dense_math)
dense_sci_acc,  _, _, _ = accuracy(dense_science)
print(f"density={density} | Math: {dense_math_acc:.2%} | Science: {dense_sci_acc:.2%} | Avg: {(dense_math_acc+dense_sci_acc)/2:.2%}")

SCIENCE: 100%|██████████| 200/200 [11:45<00:00,  3.53s/it]

density=0.5 | Math: 18.00% | Science: 36.00% | Avg: 27.00%


### 3. Try dare_linear (untested method)

In [33]:
MERGE_TYPE   = "dare_linear"
density      = 0.2
weight_math  = 1.0
weight_sci   = 0.4
weights      = [weight_math, weight_sci]
wnames       = '_'.join([str(float(w)) for w in weights])

model.add_weighted_adapter(TASK_NAMES, weights, "merge_dare_linear",
                           combination_type=MERGE_TYPE, density=density)
model.set_adapter("merge_dare_linear")

dl_math    = run_inference("MATH", math_data)
dl_science = run_inference("SCIENCE", science_data)

for task, results in [("MATH", dl_math), ("SCIENCE", dl_science)]:
    result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}/{task}"
    os.makedirs(result_dir, exist_ok=True)
    with open(f"{result_dir}/w_{wnames}_d_{density}_result.json", "w") as f:
        json.dump(results, f, indent=2)

dl_math_acc, _, _, _ = accuracy(dl_math)
dl_sci_acc,  _, _, _ = accuracy(dl_science)
print(f"dare_linear | Math: {dl_math_acc:.2%} | Science: {dl_sci_acc:.2%} | Avg: {(dl_math_acc+dl_sci_acc)/2:.2%}")

SCIENCE: 100%|██████████| 200/200 [26:57<00:00,  8.09s/it]

dare_linear | Math: 18.00% | Science: 38.50% | Avg: 28.25%


### 4. Full Summary — all methods and settings

In [34]:
import json, re

def extract_answer(response):
    matches = re.findall(r'\(([A-D])\)', response)
    return matches[-1] if matches else None

def accuracy(results):
    correct = no_pred = 0
    for r in results:
        pred = extract_answer(r["response"])
        if pred is None:
            no_pred += 1
        correct += (pred == r["answer"])
    return correct / len(results), correct, len(results), no_pred

def load(path):
    with open(path) as f:
        return json.load(f)

BASE = "/content/drive/MyDrive/merge_result/outputs"

runs = [
    ("MATH only",                   load(f"{BASE}/MATH_only/MATH/w_1.0_d_1.0_result.json"),         None),
    ("SCIENCE only",                None,                                                             load(f"{BASE}/SCIENCE_only/SCIENCE/w_1.0_d_1.0_result.json")),
    ("linear          w[1.0,0.4]",  load(f"{BASE}/linear/MATH/w_1.0_0.4_d_0.2_result.json"),        load(f"{BASE}/linear/SCIENCE/w_1.0_0.4_d_0.2_result.json")),
    ("dare_ties       w[1.0,0.4]",  load(f"{BASE}/dare_ties/MATH/w_1.0_0.4_d_0.2_result.json"),     load(f"{BASE}/dare_ties/SCIENCE/w_1.0_0.4_d_0.2_result.json")),
    ("magnitude_prune w[1.0,0.4]",  load(f"{BASE}/magnitude_prune/MATH/w_1.0_0.4_d_0.2_result.json"), load(f"{BASE}/magnitude_prune/SCIENCE/w_1.0_0.4_d_0.2_result.json")),
    ("magnitude_prune w[0.6,1.0]",  load(f"{BASE}/magnitude_prune_tuned/MATH/w_0.6_1.0_d_0.2_result.json"), load(f"{BASE}/magnitude_prune_tuned/SCIENCE/w_0.6_1.0_d_0.2_result.json")),
    ("magnitude_prune d=0.5",       load(f"{BASE}/magnitude_prune_d0.5/MATH/w_1.0_0.4_d_0.5_result.json"), load(f"{BASE}/magnitude_prune_d0.5/SCIENCE/w_1.0_0.4_d_0.5_result.json")),
    ("dare_linear     w[1.0,0.4]",  load(f"{BASE}/dare_linear/MATH/w_1.0_0.4_d_0.2_result.json"),   load(f"{BASE}/dare_linear/SCIENCE/w_1.0_0.4_d_0.2_result.json")),
]

print(f"{'Setting':<30} {'MATH':>8} {'SCIENCE':>10} {'AVG':>8}")
print("-" * 60)
for name, math_res, sci_res in runs:
    m = f"{accuracy(math_res)[0]:.2%}" if math_res else "N/A"
    s = f"{accuracy(sci_res)[0]:.2%}"  if sci_res  else "N/A"
    if math_res and sci_res:
        avg = f"{(accuracy(math_res)[0] + accuracy(sci_res)[0]) / 2:.2%}"
    else:
        avg = "N/A"
    print(f"{name:<30} {m:>8} {s:>10} {avg:>8}")

Setting                            MATH    SCIENCE      AVG
------------------------------------------------------------
MATH only                        22.00%        N/A      N/A
SCIENCE only                        N/A     62.00%      N/A
linear          w[1.0,0.4]       15.00%     23.50%   19.25%
dare_ties       w[1.0,0.4]       14.50%     43.00%   28.75%
magnitude_prune w[1.0,0.4]       26.50%     48.50%   37.50%
magnitude_prune w[0.6,1.0]       29.50%     52.50%   41.00%
magnitude_prune d=0.5            18.00%     36.00%   27.00%
dare_linear     w[1.0,0.4]       18.00%     38.50%   28.25%


### BELOW is the result before improvements

```
                              MATH    SCIENCE
---------------------------------------------
MATH adapter only           22.00%        N/A
SCIENCE adapter only           N/A     62.00%
Merged (dare_ties)          14.50%     43.00%
Average (merged)            28.75%
```

```
                              MATH    SCIENCE
---------------------------------------------
MATH adapter only           22.00%        N/A
SCIENCE adapter only           N/A     62.00%
Merged (magnitude_prune)    26.50%     48.50%
Average (merged)            37.50%


Tuned weights [0.6, 1.0] | Math: 29.50% | Science: 52.50% | Avg: 41.00%
```


```
                              MATH    SCIENCE
---------------------------------------------
MATH adapter only           22.00%        N/A
SCIENCE adapter only           N/A     62.00%
Merged (linear)             15.00%     23.50%
Average (merged)            19.25%
```


## Final Results

```
Setting                            MATH    SCIENCE      AVG
------------------------------------------------------------
MATH only                        22.00%        N/A      N/A
SCIENCE only                        N/A     62.00%      N/A
linear          w[1.0,0.4]       15.00%     23.50%   19.25%
dare_ties       w[1.0,0.4]       14.50%     43.00%   28.75%
magnitude_prune w[1.0,0.4]       26.50%     48.50%   37.50%
magnitude_prune w[0.6,1.0]       29.50%     52.50%   41.00%
magnitude_prune d=0.5            18.00%     36.00%   27.00%
dare_linear     w[1.0,0.4]       18.00%     38.50%   28.25%
```

In [ ]:
###